# Bonus 05 — LiteLLM Gateway Pattern

**Optional | After Lab 5 or Lab 7 | CPU | OpenAI API key required**

This bonus lab teaches the gateway pattern: your application talks to one stable interface, while the gateway routes requests to different model providers or deployments. In production, this is how teams add fallback models, centralize cost tracking, and avoid rewriting applications every time a backend changes.

LiteLLM supports direct Python calls and a proxy server. In this notebook we use the Python SDK path because it runs cleanly in Colab. The deployment idea is the same: model calls become provider-neutral routing decisions.

## What You Will Build

1. A direct OpenAI SDK call as the baseline.
2. The same call through LiteLLM.
3. A small routing table that makes backend choice explicit.
4. A streaming response using the same gateway function.
5. A simple cost/latency log you can extend in a real service.

> Official docs note that LiteLLM uses OpenAI-style input/output, supports retry/fallback routing, and can act as a proxy server for gateway use cases: https://docs.litellm.ai/
\n> **Note:** LiteLLM is a fast-moving library. We pin the version in the install cell to ensure the routing API works exactly as written here.\n

In [ ]:
%%capture
!pip install -q litellm openai pandas
print('Done')


In [ ]:
# Configuration — works in Colab Secrets or local environment variables
import os

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, add a secret named {name} and enable Notebook access."
        )
    return value

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
DEFAULT_MODEL = 'gpt-4o-mini'

print(f'Config loaded — OPENAI_API_KEY starts with {OPENAI_API_KEY[:8]}...')


## Part A — Baseline: Direct Client Call

Before adding a gateway, make the normal call explicit. This is what most apps start with: one provider, one model, one client. It is simple, but the provider choice is embedded directly in application code.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
messages = [
    {'role': 'system', 'content': 'You are a concise LLM deployment coach.'},
    {'role': 'user', 'content': 'When should a team add an LLM gateway? Answer in 3 bullets.'},
]

response = openai_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    temperature=0.2,
)
print(response.choices[0].message.content)


## Part B — Same Request Through LiteLLM

LiteLLM keeps the request shape familiar: `model`, `messages`, `temperature`, and the response object all look like chat-completion concepts. The model string carries the provider prefix, such as `openai/...`, `anthropic/...`, `huggingface/...`, or `ollama/...`.

In [ ]:
from litellm import completion

gateway_response = completion(
    model=f'openai/{DEFAULT_MODEL}',
    messages=messages,
    temperature=0.2,
)
print(gateway_response.choices[0].message.content)


## Part C — Make Routing a Configuration Decision

The point of a gateway is not just another library call. The point is to move backend choice out of product code. In a real deployment, this table might live in a YAML file, environment variables, or a managed gateway admin UI.

In [ ]:
import time
import pandas as pd
from dataclasses import dataclass

@dataclass
class Route:
    name: str
    model: str
    purpose: str

ROUTES = {
    'fast': Route('fast', f'openai/{DEFAULT_MODEL}', 'default low-latency classroom model'),
    'quality': Route('quality', 'openai/gpt-4o', 'higher-quality comparison model'),
}

call_log = []

def gateway_chat(route_name: str, user_prompt: str, temperature: float = 0.2):
    route = ROUTES[route_name]
    t0 = time.time()
    result = completion(
        model=route.model,
        messages=[
            {'role': 'system', 'content': 'You are a concise LLM deployment coach.'},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=temperature,
    )
    latency_ms = int((time.time() - t0) * 1000)
    usage = getattr(result, 'usage', None)
    call_log.append({
        'route': route.name,
        'model': route.model,
        'latency_ms': latency_ms,
        'prompt_tokens': getattr(usage, 'prompt_tokens', None),
        'completion_tokens': getattr(usage, 'completion_tokens', None),
    })
    return result.choices[0].message.content

print(gateway_chat('fast', 'Explain fallback routing in one paragraph.'))
pd.DataFrame(call_log)


## Part D — Streaming Through the Gateway

Streaming is where gateway compatibility matters. The app should not care whether tokens come from OpenAI, a vLLM server, or a hosted endpoint. It should consume chunks from the gateway and render partial text.

In [ ]:
stream = completion(
    model=f'openai/{DEFAULT_MODEL}',
    messages=[{'role': 'user', 'content': 'Explain LiteLLM gateways in exactly 5 short sentences.'}],
    stream=True,
)

full_text = ''
for chunk in stream:
    delta = chunk.choices[0].delta.content or ''
    full_text += delta
    print(delta, end='')

print('\n\n---')
print(f'Total characters streamed: {len(full_text)}')


## Part E — What the Proxy Adds

The Python SDK teaches the routing idea. The proxy server turns that idea into infrastructure. In a production setup, applications call the proxy through an OpenAI-compatible base URL, and the proxy handles provider credentials, virtual keys, budgets, retries, fallbacks, and observability.

Example deployment shape:

```text
Application -> OpenAI SDK base_url=http://gateway:4000/v1 -> LiteLLM Proxy -> OpenAI / vLLM / HF Endpoint / Azure / Ollama
```

Optional local command, if you want to demo the proxy outside Colab:

```bash
pip install "litellm[proxy]"
litellm --model openai/gpt-4o-mini --port 4000
```

Then client code changes only the `base_url`:

```python
client = OpenAI(api_key="anything", base_url="http://localhost:4000/v1")
```


## Lab Complete

You should now have:

- [ ] Called a model directly with the OpenAI SDK.
- [ ] Called the same model through LiteLLM.
- [ ] Used a route table to make backend choice explicit.
- [ ] Streamed tokens through the gateway abstraction.
- [ ] Logged latency and token usage for each route.

## Stretch Goals

1. Add a second provider route if you have another API key.
2. Add a retry rule: if the quality route fails, fall back to the fast route.
3. Add a simple budget guard: reject calls after total completion tokens exceed a threshold.
4. Start the LiteLLM proxy locally and call it with `OpenAI(base_url=...)`.
